In [2]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPU available: []


ResNet50-Based Palmprint Recognition
Project: Deep Learning-Based Palmprint Recognition System for Contactless Biometric Identification and Authentication of Registered Users

Model: ResNet50

Student ID: IT23136274

## 1. Environment Setup and Imports

This section imports the required libraries for implementing the ResNet50-based palmprint recognition model.

In [2]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dropout,
    Dense
)

from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed set to:", SEED)

Random seed set to: 42


## 2. ResNet50 Model Configuration

The ResNet50 model will use transfer learning with ImageNet pretrained weights. The input image size is set to 224 × 224 pixels with three RGB channels.

In [4]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_CHANNELS = 3

NUM_CLASSES = 600

INITIAL_LEARNING_RATE = 0.001
DROPOUT_RATE = 0.5

print("Image size:", (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))
print("Number of classes:", NUM_CLASSES)

Image size: (224, 224, 3)
Number of classes: 600


## 3. ResNet50 Transfer Learning Architecture

A ResNet50 model pretrained on ImageNet is used as the feature extractor. The original classification layer is removed and replaced with a custom classification head for palmprint identification.

In [5]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
)

print("ResNet50 base model loaded successfully.")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 71s 1us/step
ResNet50 base model loaded successfully.


In [6]:
base_model.trainable = False

print("Base model trainable:", base_model.trainable)

Base model trainable: False


In [7]:
x = base_model.output

x = GlobalAveragePooling2D(name="global_average_pooling")(x)

x = Dropout(
    DROPOUT_RATE,
    name="dropout"
)(x)

output = Dense(
    NUM_CLASSES,
    activation="softmax",
    name="palm_identity"
)(x)

model = Model(
    inputs=base_model.input,
    outputs=output
)

print("ResNet50 palmprint model created successfully.")

ResNet50 palmprint model created successfully.


In [8]:
model.compile(
    optimizer=Adam(learning_rate=INITIAL_LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

Model compiled successfully.


In [9]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,817,112 (94.67 MB)

 Trainable params: 1,229,400 (4.69 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

## 4. Training Callbacks

Training callbacks are configured to monitor validation performance, reduce the learning rate when improvement slows, stop training when validation performance no longer improves, and save the best-performing model.

In [10]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

# Save the best model during training
checkpoint_path = "best_resnet50_palmprint.keras"

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

print("Training callbacks configured successfully.")

Training callbacks configured successfully.


## 5. Dataset Integration

The preprocessed palmprint dataset is loaded from Google Drive. The dataset contains two sessions that will be used according to the common experimental split defined for all models.

In [13]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATASET_DIR = "/content/drive/MyDrive/SE_4050_Palm_Dataset"

print("Dataset folder exists:", os.path.exists(DATASET_DIR))

if os.path.exists(DATASET_DIR):
    print("\nAvailable files:")
    for item in os.listdir(DATASET_DIR):
        print("-", item)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import zipfile
import os

SESSION1_ZIP = os.path.join(DATASET_DIR, "session1.zip")
SESSION2_ZIP = os.path.join(DATASET_DIR, "session2.zip")

EXTRACT_DIR = "/content/palm_dataset"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(SESSION1_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

with zipfile.ZipFile(SESSION2_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extraction completed.")
print("Extracted contents:", os.listdir(EXTRACT_DIR))

In [ ]:
SESSION1_DIR = os.path.join(EXTRACT_DIR, "session1")
SESSION2_DIR = os.path.join(EXTRACT_DIR, "session2")

session1_files = sorted(os.listdir(SESSION1_DIR))
session2_files = sorted(os.listdir(SESSION2_DIR))

print("Session 1 image count:", len(session1_files))
print("Session 2 image count:", len(session2_files))

print("\nFirst 10 files in Session 1:")
print(session1_files[:10])

print("\nFirst 10 files in Session 2:")
print(session2_files[:10])